# Structured vs. Prose Context [Step 01.05]

> **MLCourse - Agentic AI - Agent Patterns**

The same facts can be handed to a model in very different shapes:

```
  PROSE                                  STRUCTURED
  -----                                  ----------
  "Order NW-10231 was placed on the      order_id: NW-10231
   3rd of October by Marta Reis and      placed:   2025-10-03
   contains a Trailhead 29er frame       customer: Marta Reis
   which shipped on the 6th..."          items:    [Trailhead 29er frame]
                                         shipped:  2025-10-06
```

Same information. Different token cost, different retrieval behaviour, different
error modes. This notebook measures the difference instead of arguing about it.

### What you'll learn

- Token cost of the same facts as prose, JSON, YAML-ish, and a Markdown table.
- Which format the model reads most reliably, measured on a lookup task.
- When prose is genuinely the better choice (it sometimes is).

### Why it matters

Structured context is the standard advice, and it is mostly right - but the
usual justification ("models like JSON") is wrong, and the usual implementation
(dump `json.dumps(obj, indent=2)`) is wasteful. The real trade-off is between
**unambiguous field boundaries** and **token overhead from punctuation**.

### Prerequisites

- [01_what_goes_in_the_window](01_what_goes_in_the_window.ipynb)

### Setup: environment, model, token counting, rate-limit-aware call helper


In [ ]:
import os                              # environment variables
import time                            # timing and pacing
import json                            # pretty-printing structured context
from pathlib import Path               # locating the track root
from dotenv import load_dotenv         # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we hit the repo root, then load the
# (gitignored) .env that lives inside 03_agentic_ai. Note the extra path
# segment: the walk-up lands on the REPO ROOT, not on the track folder.
TRACK = Path.cwd()
while not (TRACK / "03_agentic_ai").exists() and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / "03_agentic_ai" / ".env")

GROQ_MODEL = "qwen/qwen3.8-27b"        # hosted, fast, generous free tier
# Local alternative (documented, not used here): Ollama `llama3.1:8b` via
# `from langchain_ollama import ChatOllama`. OpenAI is never used in this course.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 300, **kw):
    """One place that constructs the chat model, so every notebook is identical."""
    return ChatGroq(model=GROQ_MODEL, temperature=temperature,
                    max_tokens=max_tokens, **kw)


# --- Token counting -----------------------------------------------------------
# Two different numbers, and it matters which one you are looking at:
#   * approx_tokens(): a LOCAL estimate using tiktoken's cl100k_base. It is not
#     the model's own tokenizer, so treat it as "within ~10%", good for
#     budgeting BEFORE you send a request.
#   * usage_metadata on the response: the provider's EXACT count. Ground truth,
#     but only available AFTER you have already paid for the call.
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def approx_tokens(text) -> int:
    """Approximate token count for a string (or anything str()-able)."""
    return len(_ENC.encode(str(text)))


# --- Rate-limit-aware calling --------------------------------------------------
# The Groq free tier allows 8000 tokens per minute. Several notebooks here make
# many small calls in a loop, so we self-pace well under the ceiling and retry
# with exponential backoff if we are throttled anyway.

TPM_BUDGET = 3500                       # deliberately conservative
_WINDOW = []                            # [(timestamp, tokens), ...]
USAGE = {"calls": 0, "in": 0, "out": 0, "seconds": 0.0}


def _pace(cost: int):
    """Sleep just enough that our rolling 60s token usage stays under budget."""
    now = time.time()
    while True:
        recent = [(t, n) for (t, n) in _WINDOW if now - t < 60]
        _WINDOW[:] = recent
        if sum(n for _, n in recent) + cost <= TPM_BUDGET or not recent:
            return
        time.sleep(min(5.0, 60 - (now - recent[0][0]) + 0.5))
        now = time.time()


def chat(messages, llm=None, temperature=0.0, max_tokens=300, retries=5):
    """Send `messages`, return the AIMessage. Paces, retries, and meters usage.

    `messages` is a list of (role, content) tuples or LangChain message objects.
    """
    llm = llm or make_llm(temperature=temperature, max_tokens=max_tokens)
    est = approx_tokens(messages) + max_tokens
    delay = 4.0
    for attempt in range(retries):
        _pace(est)
        t0 = time.time()
        try:
            out = llm.invoke(messages)
        except Exception as exc:
            if "rate_limit" in str(exc) or "429" in str(exc):
                time.sleep(delay)
                delay = min(delay * 2, 45)
                continue
            raise
        u = out.usage_metadata or {}
        _WINDOW.append((time.time(), u.get("total_tokens", est)))
        USAGE["calls"] += 1
        USAGE["in"] += u.get("input_tokens", 0)
        USAGE["out"] += u.get("output_tokens", 0)
        USAGE["seconds"] += time.time() - t0
        return out
    raise RuntimeError("still rate limited after %d attempts" % retries)


def ask(prompt: str, system: str = None, **kw) -> str:
    """Convenience wrapper: one user turn in, plain text out."""
    msgs = ([("system", system)] if system else []) + [("user", prompt)]
    return chat(msgs, **kw).content.strip()


print("model:", GROQ_MODEL)
print("key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("tokenizer:", "cl100k_base (approximation)")


### 1. One dataset, four renderings

Five order records. We will render them four ways and measure each.

In [2]:
ORDERS = [
    {"order_id": "NW-10231", "customer": "Marta Reis", "placed": "2025-10-03",
     "item": "Trailhead 29er frame", "status": "shipped", "total_eur": 899},
    {"order_id": "NW-10244", "customer": "Ben Kowalski", "placed": "2025-10-05",
     "item": "Cirrus road wheelset", "status": "processing", "total_eur": 640},
    {"order_id": "NW-10250", "customer": "Aisha Farouk", "placed": "2025-10-06",
     "item": "Nimbus commuter bike", "status": "delivered", "total_eur": 1180},
    {"order_id": "NW-10261", "customer": "Jonas Weber", "placed": "2025-10-08",
     "item": "Trailhead 27.5 frame", "status": "cancelled", "total_eur": 849},
    {"order_id": "NW-10277", "customer": "Priya Nair", "placed": "2025-10-09",
     "item": "Vector gravel bike", "status": "shipped", "total_eur": 1490},
]


def as_prose(rows):
    out = []
    for r in rows:
        out.append("Order %s was placed on %s by %s. It contains one %s, the "
                   "order total is %d euros, and its current status is %s."
                   % (r["order_id"], r["placed"], r["customer"], r["item"],
                      r["total_eur"], r["status"]))
    return " ".join(out)


def as_json(rows):
    return json.dumps(rows, indent=2)


def as_json_compact(rows):
    return json.dumps(rows, separators=(",", ":"))


def as_keyvalue(rows):
    blocks = []
    for r in rows:
        blocks.append("\n".join("%s: %s" % (k, v) for k, v in r.items()))
    return "\n---\n".join(blocks)


def as_table(rows):
    cols = list(rows[0])
    head = "| " + " | ".join(cols) + " |"
    sep = "|" + "|".join("---" for _ in cols) + "|"
    body = ["| " + " | ".join(str(r[c]) for c in cols) + " |" for r in rows]
    return "\n".join([head, sep] + body)


FORMATS = {
    "prose":         as_prose(ORDERS),
    "json (indent)": as_json(ORDERS),
    "json (compact)": as_json_compact(ORDERS),
    "key: value":    as_keyvalue(ORDERS),
    "markdown table": as_table(ORDERS),
}

print(FORMATS["markdown table"])
print()
print(FORMATS["key: value"][:160], "...")

| order_id | customer | placed | item | status | total_eur |
|---|---|---|---|---|---|
| NW-10231 | Marta Reis | 2025-10-03 | Trailhead 29er frame | shipped | 899 |
| NW-10244 | Ben Kowalski | 2025-10-05 | Cirrus road wheelset | processing | 640 |
| NW-10250 | Aisha Farouk | 2025-10-06 | Nimbus commuter bike | delivered | 1180 |
| NW-10261 | Jonas Weber | 2025-10-08 | Trailhead 27.5 frame | cancelled | 849 |
| NW-10277 | Priya Nair | 2025-10-09 | Vector gravel bike | shipped | 1490 |

order_id: NW-10231
customer: Marta Reis
placed: 2025-10-03
item: Trailhead 29er frame
status: shipped
total_eur: 899
---
order_id: NW-10244
customer: Ben Kowals ...


In [3]:
base = approx_tokens(FORMATS["prose"])
print("%-16s %8s %10s" % ("format", "tokens", "vs prose"))
print("-" * 38)
for name, text in sorted(FORMATS.items(), key=lambda kv: approx_tokens(kv[1])):
    n = approx_tokens(text)
    print("%-16s %8d %9.0f%%" % (name, n, 100 * n / base))

format             tokens   vs prose
--------------------------------------
markdown table        180        80%
json (compact)        219        97%
key: value            224        99%
prose                 226       100%
json (indent)         329       146%


### The first surprise

Indented JSON is usually the **most expensive** rendering, often by a wide
margin, and every one of those extra tokens is whitespace and punctuation. The
Markdown table is typically the cheapest because it writes each field name once
instead of once per record.

That last point is the actual rule:

> Repeated field names are the tax. Any format that names the columns once beats
> any format that names them per row.

This is why a table wins for **homogeneous** records and loses for
**heterogeneous** ones (where most cells would be empty).

### 2. Which format does the model read most accurately?

Token cost is only half the question. Now we test comprehension: five lookup
questions per format, all answerable from the data, some requiring a small
cross-field join.

In [4]:
QUESTIONS = [
    ("What is the status of order NW-10261?", "cancelled"),
    ("Which customer placed order NW-10250?", "aisha"),
    ("What is the total in euros for Priya Nair's order?", "1490"),
    ("How many orders have the status 'shipped'?", "2"),
    ("Which order was placed earliest?", "10231"),
]

SYS = ("Answer using ONLY the order data provided. "
       "Reply with the shortest possible answer - a single word or number.")

scores = {}
for fmt_name, text in FORMATS.items():
    hits = []
    for q, expected in QUESTIONS:
        out = chat([("system", SYS),
                    ("user", "Order data:\n%s\n\nQuestion: %s" % (text, q))],
                   temperature=0.0, max_tokens=32)
        a = out.content.strip().lower()
        hits.append(expected in a)
    scores[fmt_name] = hits
    print("%-16s %s  %d/%d" % (fmt_name,
                               "".join("O" if h else "." for h in hits),
                               sum(hits), len(hits)))

prose            OOOOO  5/5


json (indent)    OOOOO  5/5


json (compact)   OOOOO  5/5


key: value       OOOOO  5/5


markdown table   OOOOO  5/5


In [5]:
print("%-16s %8s %10s %12s" % ("format", "tokens", "accuracy", "tokens/correct"))
print("-" * 52)
for name in sorted(FORMATS, key=lambda n: -sum(scores[n])):
    n_tok = approx_tokens(FORMATS[name])
    correct = sum(scores[name])
    eff = n_tok / correct if correct else float("inf")
    print("%-16s %8d %9.0f%% %12.0f" % (name, n_tok,
                                        100 * correct / len(QUESTIONS), eff))

format             tokens   accuracy tokens/correct
----------------------------------------------------
prose                 226       100%           45
json (indent)         329       100%           66
json (compact)        219       100%           44
key: value            224       100%           45
markdown table        180       100%           36


> **Read this honestly.** Five questions per format is a demonstration, not a
> benchmark - the accuracy column will often be all-100% on a task this easy, and
> when it is, the only column that separates the formats is **tokens**. That is
> itself the finding: *when comprehension is equal, pick the cheapest rendering*.
> If you want a real answer for your own data, run this same loop with fifty
> questions against your actual records.

### 3. Where prose actually wins

Structured wins on lookup. Prose wins on three specific things, and they are not
edge cases:

| Prose is better for | Why |
|---|---|
| **Relationships and causation** | "The refund was delayed *because* the item arrived damaged" has no natural column |
| **Uncertainty and hedging** | "probably", "the customer seemed to imply" - a boolean field would be a lie |
| **Narrative order** | What happened, in what order, and what it means |

The practical rule is a **hybrid**: facts as a table, context as a sentence.
Let us measure whether that hybrid actually answers a causal question the table
cannot.

In [6]:
NOTE = ("Context: NW-10261 was cancelled because the frame size Jonas ordered "
        "was discontinued mid-order; he is waiting on a restock rather than a refund.")

q = "Should we issue Jonas Weber a refund? Answer yes or no and give one reason."

table_only = chat([("system", SYS.replace("a single word or number", "one short sentence")),
                   ("user", "Order data:\n%s\n\nQuestion: %s" % (FORMATS["markdown table"], q))],
                  temperature=0.0, max_tokens=90).content.strip()

hybrid = chat([("system", SYS.replace("a single word or number", "one short sentence")),
               ("user", "Order data:\n%s\n\n%s\n\nQuestion: %s"
                        % (FORMATS["markdown table"], NOTE, q))],
              temperature=0.0, max_tokens=90).content.strip()

print("TABLE ONLY (%d ctx tokens):" % approx_tokens(FORMATS["markdown table"]))
print(" ", table_only)
print()
print("TABLE + PROSE NOTE (%d ctx tokens):"
      % approx_tokens(FORMATS["markdown table"] + NOTE))
print(" ", hybrid)
print()
print("the note costs %d extra tokens" % approx_tokens(NOTE))

TABLE ONLY (180 ctx tokens):
  Yes, because his order was cancelled.

TABLE + PROSE NOTE (211 ctx tokens):
  No, because he is waiting on a restock rather than a refund.

the note costs 31 extra tokens


The table can only report `status: cancelled`. It has no column for *why*, and
inventing one (`cancellation_reason_code`) would still lose "he is waiting on a
restock rather than a refund". That nuance is what prose is for.

### 4. A practical checklist

When you next assemble context, run down this list:

- **Homogeneous records?** -> Markdown table. Cheapest, and boundaries are clear.
- **One nested object?** -> `key: value` lines, not indented JSON.
- **Must round-trip to code?** -> compact JSON (`separators=(",", ":")`),
  never `indent=2`.
- **Explaining, hedging, or causal?** -> prose, deliberately.
- **Both?** -> table for the facts, one sentence of prose for the context. That
  is the normal answer.

And one formatting rule that matters more than the format choice:

> **Label your blocks.** `Order data:` / `Reference material:` / `Conversation so
> far:` costs about three tokens and prevents the model from confusing retrieved
> text with instructions - which is also the cheapest prompt-injection mitigation
> you will ever deploy (see `05_production_security/01_prompt_injection`).

### 5. Pitfalls

- **`json.dumps(obj, indent=2)` by reflex.** You are paying for whitespace on
  every call, forever.
- **Tables for sparse data.** If half the cells are empty, the table is a lie
  wearing a grid; use key/value blocks and omit the missing keys.
- **Structuring away the nuance.** Forcing a hedge into a boolean loses the
  hedge. If the fact is uncertain, the context should say so in words.
- **Unlabelled blocks.** Context that is not introduced gets read as instruction.

### Recap

| Idea | Takeaway |
|---|---|
| Repeated keys are the tax | Name columns once; tables beat per-row JSON |
| Indented JSON is the worst | Whitespace is billable |
| Measure comprehension too | Cheapest format is only better if accuracy holds |
| Prose for causation | Relationships, hedging and narrative have no column |
| Label every block | Three tokens, big return |

### Module complete

You can now measure what is in a context window, budget it, trim it three
different ways, order it for how models actually attend, and format it for cost
and comprehension.

**Next module:** [02_memory_at_scale](../02_memory_at_scale) - what to do when
the conversation itself outgrows every budget you set.